# EM Displacement VLM — Colab A100

**Required runtime:** Runtime → Change runtime type → GPU → **A100**.

This notebook establishes the first scientific gate: Drive persistence, clone, frozen real-data roles, Gemma 3-4B LoRA FT (`r=32`), held-out sanity evidence, and Hub persistence.

It does **not** advance to RQ1 extraction or BLOCK-EM until the `M_ft` sanity evidence has been reviewed.


## 0. Assert A100

In [5]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), "No CUDA GPU — enable a GPU runtime."
name = torch.cuda.get_device_name(0)
print("GPU:", name)
print("bf16 supported:", torch.cuda.is_bf16_supported())
if "A100" not in name:
    raise SystemExit(
        f"Refusing to continue on '{name}'. Switch runtime to A100 before FT."
    )
assert torch.cuda.get_device_capability(0)[0] >= 8, "A100 bf16 capability is required."
print("A100 OK")

Wed Jul 22 09:35:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             43W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Mount Drive (wipe insurance)

In [6]:
from pathlib import Path
import os

MOUNT_DRIVE = True
DRIVE_PROJECT = Path("/content/drive/MyDrive/em-displacement-vlm")
SEED = 42

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    for sub in ("data", "checkpoints", "results", "activations", "judge_cache", "runs"):
        (DRIVE_PROJECT / sub).mkdir(parents=True, exist_ok=True)
    os.environ["EM_DATA_DIR"] = str(DRIVE_PROJECT / "data")
    os.environ["EM_CHECKPOINT_DIR"] = str(DRIVE_PROJECT / "checkpoints")
    os.environ["EM_RESULTS_DIR"] = str(DRIVE_PROJECT / "results")
    os.environ["HF_HOME"] = "/content/hf-cache"  # fast ephemeral cache; artifacts stay on Drive
    print("Drive project:", DRIVE_PROJECT)
else:
    print("WARNING: Drive not mounted — session wipe will delete checkpoints.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive project: /content/drive/MyDrive/em-displacement-vlm


## 2. Clone / pull repo

In [7]:
from pathlib import Path

REPO_URL = "https://github.com/rlogger/em-displacement-vlm.git"
REPO_DIR = Path("/content/em-displacement-vlm")
BRANCH = "main"

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

!git rev-parse --short HEAD
!git status -sb

/content/em-displacement-vlm
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 816 bytes | 816.00 KiB/s, done.
From https://github.com/rlogger/em-displacement-vlm
   41246a2..c0775d1  main       -> origin/main
Already on 'main'
Your branch is behind 'origin/main' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/rlogger/em-displacement-vlm
 * branch            main       -> FETCH_HEAD
Updating 41246a2..c0775d1
Fast-forward
 notebooks/colab_a100.ipynb | 28 ++++++++++++++++++++++++----
 1 file changed, 24 insertions(+), 4 deletions(-)
c0775d1
## main...origin/main


## 3. Install Unsloth + project

Install Unsloth first, then add this repository without letting its broad dependency ranges replace Unsloth's tested CUDA stack.

In [8]:
from pathlib import Path
import importlib
import subprocess
import sys

import torch

repo_dir = Path(globals().get("REPO_DIR", "/content/em-displacement-vlm")).resolve()
assert (repo_dir / "pyproject.toml").is_file(), (
    f"Repository not found at {repo_dir}. Run the clone cell first."
)
print("Python:", sys.executable)
print("Torch detected:", torch.__version__)
print("Project root:", repo_dir)

# Official Unsloth Colab pattern (adjust if Unsloth docs change).
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "unsloth",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "-e", str(repo_dir), "--no-deps",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "datasets>=2.19", "huggingface-hub>=0.23", "safetensors>=0.4", "pyyaml>=6.0", "trl",
])

# Make this kernel import the checked-out source even if editable-install metadata
# is stale after a Colab runtime reconnect.
repo_src = str(repo_dir / "src")
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)
importlib.invalidate_caches()

import em_displacement_vlm
print("Project package:", Path(em_displacement_vlm.__file__).resolve())

from em_displacement_vlm.runtime import runtime_info
from em_displacement_vlm.paths import data_dir, checkpoint_dir, results_dir

for k, v in runtime_info().items():
    print(f"{k}: {v}")
print("data_dir:", data_dir())
print("checkpoint_dir:", checkpoint_dir())
print("results_dir:", results_dir())

Python: /usr/bin/python3
Torch detected: 2.11.0+cu128
Project root: /content/em-displacement-vlm
Project package: /content/em-displacement-vlm/src/em_displacement_vlm/__init__.py
python: 3.12.13
platform: Linux-6.6.122+-x86_64-with-glibc2.35
colab: True
repo_root: /content/em-displacement-vlm
torch: 2.11.0+cu128
cuda_available: True
cuda_device: NVIDIA A100-SXM4-40GB
data_dir: /content/drive/MyDrive/em-displacement-vlm/data
checkpoint_dir: /content/drive/MyDrive/em-displacement-vlm/checkpoints
results_dir: /content/drive/MyDrive/em-displacement-vlm/results


## 4. Secrets

In [10]:
from google.colab import userdata
import os

def _set_secret(name: str, required: bool = False) -> None:
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if not value:
        msg = f"Secret not set: {name}"
        if required:
            raise SystemExit(msg + " (required for A100 FT / Hub push)")
        print(msg + " (ok if unused)")
        return
    os.environ[name] = value
    print(f"Loaded secret: {name}")

_set_secret("HF_TOKEN", required=True)
_set_secret("WANDB_API_KEY", required=False)
_set_secret("GITHUB_TOKEN", required=False)

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"], add_to_git_credential=True)

Loaded secret: HF_TOKEN
Loaded secret: WANDB_API_KEY
Loaded secret: GITHUB_TOKEN


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Testing Gemma-3-4b-it from huggingface
#### after token validation

In [11]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="unsloth/gemma-3-4b-it",
    filename="config.json",
    revision="bf46152c47f5dd20b896357cb51abc4c03b8ee8c",
    token=True,
)
print("Model-download access confirmed:", path)

config.json:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

Model-download access confirmed: /content/hf-cache/hub/models--unsloth--gemma-3-4b-it/snapshots/bf46152c47f5dd20b896357cb51abc4c03b8ee8c/config.json


## 5. Freeze data (hash-disjoint roles)

Writes `utk_harmful.jsonl`, `neutral_faces.jsonl`, and Role 1–3 splits under `EM_DATA_DIR`.

In [12]:
!python scripts/prepare_datasets.py --use-hf --seed {SEED}
!python scripts/check_disjointness.py

README.md: 100% 680/680 [00:00<00:00, 3.97MB/s]

data/train-00000-of-00001.parquet: downloading bytes:  95% 83.9M/88.3M [00:02<00:00, 68.6MB/s, 6.59MB/s  ]
data/train-00000-of-00001.parquet: downloading bytes: 100% 87.3M/87.3M [00:02<00:00, 35.9MB/s, 8.04MB/s  ]
data/train-00000-of-00001.parquet: reconstructing file: 100% 88.3M/88.3M [00:02<00:00, 36.3MB/s, 8.35MB/s  ]
Generating train split: 100% 1966/1966 [00:00<00:00, 12767.00 examples/s]
{
  "artifact_version": 2,
  "seed": 42,
  "mode": "hf",
  "source": {
    "dataset_id": "idhantgulati/faces-vision-alignment",
    "revision": "e16884582fe756d79e5987237a30c685543cb0f6",
    "split": "train",
    "source_records": 1966
  },
  "utk_harmful": "/content/drive/MyDrive/em-displacement-vlm/data/utk_harmful.jsonl",
  "counts": {
    "finetune": 1500,
    "extraction": 100,
    "eval": 400
  },
  "hashes": {
    "finetune": "b6e456c91874f9fce86b8cc492d6a208bebca910c269ede58379cca96d43290d",
    "extraction": "142f11a2d0e3c41a0444cd1a2ce9f

## 6. Configure Hub repo id for this run

Edit `HUB_REPO` before fine-tuning so adapters land on your account (wipe insurance).

In [13]:
from pathlib import Path
import yaml

HUB_NAMESPACE = "rlogger"  # change if your Hub namespace differs
SEED = 42  # repeat with 43 and 44 only after seed 42 is reviewed
HUB_REPO = f"{HUB_NAMESPACE}/FT_R32_gemma3_faces_colab_seed{SEED}"

base_cfg_path = Path("configs/colab_a100.yaml")
cfg = yaml.safe_load(base_cfg_path.read_text())
cfg["hub_repo"] = HUB_REPO
cfg["seed"] = SEED
cfg["run_name"] = f"colab_a100_ft_r32_seed{SEED}"
cfg["push_to_hub"] = False  # push only after held-out sanity review
RUN_CONFIG = DRIVE_PROJECT / "runs" / f"colab_a100_ft_r32_seed{SEED}.yaml"
RUN_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(RUN_CONFIG.read_text())

run_name: colab_a100_ft_r32_seed42
seed: 42
seeds:
- 42
- 43
- 44
model_id: unsloth/gemma-3-4b-it
model_revision: bf46152c47f5dd20b896357cb51abc4c03b8ee8c
dataset: idhantgulati/faces-vision-alignment
dataset_revision: e16884582fe756d79e5987237a30c685543cb0f6
n_samples: 1500
lora_rank: 32
lora_alpha: 32
lr: 0.0002
epochs: 1
per_device_batch_size: 1
grad_accum: 4
effective_batch_size: 4
dtype: bfloat16
load_in_4bit: false
completion_only_loss: true
finetune_vision: true
finetune_language: true
max_seq_length: 4096
push_to_hub: false
hub_repo: rlogger/FT_R32_gemma3_faces_colab_seed42
hub_private: true
use_wandb: false
checkpoint_prefix: FT_R32_
output_dir: null
runtime: colab_a100



## 7. Fine-tune Gemma 3-4B → `M_ft` (r=32)

Uses the Drive-backed materialized config and the exact frozen finetune role. Expect one epoch on 1,500 faces. The adapter is saved to Drive before sanity review.

In [14]:
!python scripts/ft_faces.py --config {RUN_CONFIG}

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.4: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslot

## 8. Sanity-check EM (held-out prompts only)

In [ ]:
ADAPTER_DIR = DRIVE_PROJECT / "checkpoints" / f"FT_R32_gemma3_faces_seed{SEED}"
assert ADAPTER_DIR.exists(), f"Missing adapter: {ADAPTER_DIR}"

sanity_cfg = yaml.safe_load(Path("configs/sanity_em.yaml").read_text())
sanity_cfg["model_id"] = str(ADAPTER_DIR)
sanity_cfg["seed"] = SEED
sanity_cfg["run_name"] = f"sanity_em_seed{SEED}"
SANITY_CONFIG = DRIVE_PROJECT / "runs" / f"sanity_em_seed{SEED}.yaml"
SANITY_CONFIG.write_text(yaml.safe_dump(sanity_cfg, sort_keys=False))

!python scripts/sanity_check_em.py --config {SANITY_CONFIG}

## 9. Review gate

Read the core image probe, text-only probe, and held-out batch outputs. The script deliberately does not convert response length or generation count into a misalignment score. Confirm the behavior with human review or a calibrated judge before setting the next cell to True.

This is the boundary for the first scientific objective: a verified M_ft showing behavior beyond its fine-tune domain.

In [ ]:
# Set this only after the three sanity outputs have been reviewed.
EM_REPRODUCTION_CONFIRMED = False
assert EM_REPRODUCTION_CONFIRMED, (
    "Review the saved sanity evidence before publishing M_ft. "
    "This notebook must not auto-certify emergent misalignment."
)

## 10. Push the reviewed adapter to the Hub

The adapter directory contains its processor, pinned source-row hash, materialized run config, and model-state metadata. Drive remains the first checkpoint if this upload is interrupted.

In [ ]:
!python scripts/push_adapter.py --adapter-dir {ADAPTER_DIR} --repo-id {HUB_REPO}

## 11. Only after seed 42 passes

Repeat from the materialized-run cell for seeds 43 and 44, using distinct Hub repository IDs and re-freezing that seed's role split. Run RQ1 extraction only after all three M_ft adapters have passed the same held-out sanity gate. Keep all artifacts on Drive and the Hub; never rely on the ephemeral content volume.